# 04 · Tree models, prediction intervals and a first stopping rule

The Ch 6–7 drills (HOML_exercises/ch06) explored single trees, extrapolation, forests, boosting and quantile intervals. This notebook brings the useful parts into the main repo and moves the project on to **parts 2 and 3 of the objective**:

> How trustworthy is an early-life prediction for a new batch of cells, and can it safely decide when to stop a cycling test?

| Section | Question |
|---|---|
| 1 | Tree models in the results table, and on a new batch (leave-one-batch-out) |
| 2 | Which features matter? Permutation importance on held-out folds, twins permuted together |
| 3 | 80% intervals on the paper's split: coverage, with misses split into **dangerous** (below) and harmless (above) |
| 4 | Do the intervals flag b2c1 and the long-lived batch 3 cells? |
| 5 | **Batch-calibrated conformal**: do intervals built from batch-to-batch errors survive a new batch? |
| 6 | A novelty flag: is this cell unlike anything in training? |
| 7 | **First stopping rule**: cycler time saved vs dangerous misses |

**Which misses are dangerous?** A stopping rule ends a test at cycle 100 and reports the interval. If the cell then dies **before** the lower bound (true life below the interval, like b2c1), the rule promised life the cell didn't have. If it lives **beyond** the upper bound, the rule was pessimistic: wasteful but safe.

**Rules carried over:** log₁₀ life target, % errors in cycles; CV grouped by charge policy; models **chosen by CV**; test sets only scored.

*Reading: Hands-On ML Ch 6 and 7 (done in the drills).*

The leave-one-batch-out cells take a couple of minutes.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.features import FEATURE_COLUMNS, load_feature_table
from src.evaluate import (BATCHES, evaluate, grouped_cv, pct_error,
                          save_metrics, signed_pct)

FIGURES = ROOT / "results" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
METRICS_CSV = ROOT / "results" / "metrics.csv"
SEED = 42
F = FEATURE_COLUMNS

data = load_feature_table(ROOT / "data" / "processed" / "features.csv")
y = np.log10(data["cycle_life"])
train = data[data["split"] == "train"]
X_tr, y_tr, groups_tr = train[F], np.log10(train["cycle_life"]), train["charge_policy"]
cv = grouped_cv(5)
MAE = "neg_mean_absolute_error"
COLS = ["cv_log_mae", "train_pct", "test_pct", "secondary_test_pct", "b1_bias_pct", "b2_bias_pct", "b3_bias_pct"]

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def lin_pipe(model):
    """Linear models: impute, scale, model (as in notebook 03)."""
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), model)


def tree_pipe(model):
    """Tree models: impute, model. No scaler: splits don't care about units (Ch 6)."""
    return make_pipeline(SimpleImputer(strategy="median"), model)


def short(params):
    """{'randomforestregressor__max_features': 0.6} -> {'max_features': 0.6}"""
    return {k.split("__")[1]: (float(np.round(v, 4)) if isinstance(v, float) else v)
            for k, v in params.items()}


# Reference: best linear model from notebook 03 (chosen there by grouped CV)
EN_BEST = lin_pipe(ElasticNet(alpha=10 ** -2.2, l1_ratio=0.7, max_iter=50_000))  # alpha ≈ 0.0063
rows = []
row, en_fit = evaluate("Elastic net", EN_BEST, data, F, note="{'alpha': 0.0063, 'l1_ratio': 0.7}")
rows.append(row)
print(data["split"].value_counts().to_dict())

## 1 · Tree models in the results table

The three tree families from the drills, each tuned by grouped CV on the 41 training cells and added to `results/metrics.csv` next to the notebook 03 models. The grids are small: the drills showed CV is flat within noise across most settings.

In [ ]:
searches = {
    "Decision tree": (tree_pipe(DecisionTreeRegressor(random_state=SEED)),
                      {"decisiontreeregressor__max_depth": [2, 3, 4, 5, None],
                       "decisiontreeregressor__min_samples_leaf": [1, 3, 5]}),
    "Random forest": (tree_pipe(RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)),
                      {"randomforestregressor__max_features": [1.0, 0.6, 0.3],
                       "randomforestregressor__min_samples_leaf": [1, 2, 4]}),
    "Gradient boosting": (tree_pipe(GradientBoostingRegressor(subsample=0.8, random_state=SEED)),
                          {"gradientboostingregressor__learning_rate": [0.03, 0.1],
                           "gradientboostingregressor__n_estimators": [50, 100, 300],
                           "gradientboostingregressor__max_depth": [1, 2, 3]}),
}
tuned = {}
for name, (model, grid) in searches.items():
    s = GridSearchCV(model, grid, cv=cv, scoring=MAE).fit(X_tr, y_tr, groups=groups_tr)
    tuned[name] = s.best_estimator_
    row, _ = evaluate(name, s.best_estimator_, data, F, note=str(short(s.best_params_)))
    rows.append(row)
    print(f"{name:18s} {short(s.best_params_)}  CV log-MAE {-s.best_score_:.4f}")

table = save_metrics(rows, METRICS_CSV)
table.loc[[r["model"] for r in rows], COLS].round(3)

In [ ]:
# Leave-one-batch-out for each family (hyperparameters re-tuned inside the two training batches)
EN_GRID = {"elasticnet__alpha": np.logspace(-4, 0, 21), "elasticnet__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]}
families = {"Elastic net": (lin_pipe(ElasticNet(max_iter=50_000)), EN_GRID),
            "Random forest": searches["Random forest"],
            "Gradient boosting": searches["Gradient boosting"]}
lobo = []
for held_out in BATCHES:
    tr_m = data["batch"] != held_out
    for name, (model, grid) in families.items():
        s = GridSearchCV(model, grid, cv=cv, scoring=MAE)
        s.fit(data.loc[tr_m, F], y[tr_m], groups=data.loc[tr_m, "charge_policy"])
        p = s.predict(data.loc[~tr_m, F])
        lobo.append({"held_out": held_out, "model": name,
                     "abs_err_pct": pct_error(y[~tr_m], p).mean(),
                     "bias_pct": signed_pct(y[~tr_m], p).mean()})
lobo = pd.DataFrame(lobo)
lobo.pivot(index="held_out", columns="model", values=["abs_err_pct", "bias_pct"]).round(1)

**Question:** does the main repo reproduce the drill conclusion (regularised linear still best by CV; tree ensembles no better on test)? In leave-one-batch-out, are the tree models' batch biases smaller or larger than the elastic net's? Use the drills' extrapolation result to explain the b2 column: what is the shortest life a tree model trained on b1 + b3 can predict, and how does that compare with b2's lives?

*Your notes:*

## 2 · Which features matter? Permutation importance done properly

The drills found two traps:
- **impurity importance** ranked a random-noise column 10th of 13, because it counts overfitted splits as importance;
- permuting **one twin** at a time under-counts the pair, because the other twin still carries the information.

So here: permutation importance on **held-out grouped CV folds** (never the cells the model was fitted on), with each **twin pair shuffled together**, and a **random-noise column** as a control. Anything not clearly above the noise bar is indistinguishable from nothing. Done for the elastic net and the best tree model, to see whether they rely on the same information.

In [ ]:
rng = np.random.default_rng(SEED)
X_noise = X_tr.copy()
X_noise["noise (control)"] = rng.normal(size=len(X_noise))

TWINS = {"ΔQ twins (var + min)": ["log_var_dq", "log_abs_min_dq"],
         "capacity twins (qd_cycle2 + fade_intercept)": ["qd_cycle2", "fade_intercept"]}
in_twins = {c for cols in TWINS.values() for c in cols}
PERM_GROUPS = {**TWINS, **{f: [f] for f in F if f not in in_twins}, "noise (control)": ["noise (control)"]}


def heldout_perm_importance(model, X, y_, groups, n_repeats=20):
    """Mean increase in held-out log-MAE when a group of columns is shuffled together."""
    gen = np.random.default_rng(SEED)
    out = {g: [] for g in PERM_GROUPS}
    for tr_i, va_i in cv.split(X, y_, groups):
        m = clone(model).fit(X.iloc[tr_i], y_.iloc[tr_i])
        Xv, yv = X.iloc[va_i], y_.iloc[va_i].to_numpy()
        base = np.abs(m.predict(Xv) - yv).mean()
        for g, cols in PERM_GROUPS.items():
            for _ in range(n_repeats):
                Xp = Xv.copy()
                Xp[cols] = Xv[cols].to_numpy()[gen.permutation(len(Xv))]
                out[g].append(np.abs(m.predict(Xp) - yv).mean() - base)
    return pd.Series({g: np.mean(v) for g, v in out.items()})


best_tree_name = min(["Random forest", "Gradient boosting"],
                     key=lambda n: table.loc[n, "cv_log_mae"])
imp = pd.DataFrame({
    "Elastic net": heldout_perm_importance(EN_BEST, X_noise, y_tr, groups_tr),
    best_tree_name: heldout_perm_importance(clone(tuned[best_tree_name]), X_noise, y_tr, groups_tr),
}).sort_values("Elastic net")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, col in zip(axes, imp.columns):
    ax.barh(imp.index, imp[col], color=["0.6" if i == "noise (control)" else "C0" for i in imp.index])
    ax.axvline(max(imp.loc["noise (control)", col], 0), color="C3", ls="--", lw=1, label="noise level")
    ax.axvline(0, color="0.5", lw=0.8)
    ax.set(title=col, xlabel="increase in held-out log-MAE when shuffled")
axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "04_permutation_importance.png", dpi=200)
imp.iloc[::-1].round(4)

**Question:** which feature groups are clearly above the noise control for each model? Do the elastic net and the tree model rely on the same information? Compare the ΔQ twins' importance here with the drills' one-at-a-time result. With only ~8 cells per validation fold, how much would you trust the ranks below the top three or four?

*Your notes:*

## 3 · 80% prediction intervals on the paper's split

| Method | How | Width |
|---|---|---|
| **EN + policy conformal** | Elastic net. Half-width = 80th percentile of its out-of-fold errors under **policy-grouped** CV on the training cells | Same for every cell |
| **EN + batch conformal** | Same, but the out-of-fold errors come from **leave-one-batch-out** inside the training cells: train on one batch, predict the other. So the errors include batch shift | Same for every cell |
| **RF tree spread** | 10th–90th percentile of the forest's individual tree predictions | Per cell |
| **Quantile boosting** | Two extra boosting models fitted to the 10th and 90th percentiles (`loss="quantile"`) | Per cell |

The quantile models are fitted separately from the main one, so they can cross it; the code clamps the interval to contain the prediction. Watch for intervals with one side of zero width.

Checks per split:
- **coverage** (target 80%), split into **below** (dangerous: cell died sooner than the lower bound) and **above** (harmless). A well-calibrated 80% interval misses about 10% each side;
- **half-width** (± % of the prediction);
- **ρ**, the rank correlation between width and actual error (does it widen where it's wrong?).

In [ ]:
LEVEL = 0.80
LO, HI = (1 - LEVEL) / 2, 1 - (1 - LEVEL) / 2   # 0.1, 0.9
RF_BEST, GB_BEST = tuned["Random forest"], tuned["Gradient boosting"]


def conformal_halfwidth(model, X, y_, groups, splitter, level=LEVEL):
    """Conformal half-width: the level-quantile of out-of-fold absolute errors."""
    oof = cross_val_predict(clone(model), X, y_, groups=groups, cv=splitter)
    r = np.sort(np.abs(np.asarray(y_) - oof))
    k = int(np.ceil((len(r) + 1) * level))
    return r[min(k, len(r)) - 1]


def intervals(train_df, pred_df):
    """80% intervals (log10 life) for every cell in pred_df, from models fitted on train_df only."""
    Xt, yt = train_df[F], np.log10(train_df["cycle_life"])
    Xp = pred_df[F]
    out = {}

    mid = clone(EN_BEST).fit(Xt, yt).predict(Xp)
    q_pol = conformal_halfwidth(EN_BEST, Xt, yt, train_df["charge_policy"], cv)
    out["EN + policy conformal"] = pd.DataFrame({"lo": mid - q_pol, "mid": mid, "hi": mid + q_pol}, index=pred_df.index)
    q_bat = conformal_halfwidth(EN_BEST, Xt, yt, train_df["batch"], LeaveOneGroupOut())
    out["EN + batch conformal"] = pd.DataFrame({"lo": mid - q_bat, "mid": mid, "hi": mid + q_bat}, index=pred_df.index)

    rf = clone(RF_BEST).fit(Xt, yt)
    per_tree = np.stack([t.predict(rf[:-1].transform(Xp)) for t in rf[-1].estimators_])
    out["RF tree spread"] = pd.DataFrame({"lo": np.quantile(per_tree, LO, axis=0), "mid": per_tree.mean(axis=0),
                                          "hi": np.quantile(per_tree, HI, axis=0)}, index=pred_df.index)

    mid = clone(GB_BEST).fit(Xt, yt).predict(Xp)
    q = {name: clone(GB_BEST).set_params(gradientboostingregressor__loss="quantile",
                                         gradientboostingregressor__alpha=a).fit(Xt, yt).predict(Xp)
         for name, a in [("lo", LO), ("hi", HI)]}
    out["Quantile boosting"] = pd.DataFrame({"lo": np.minimum(q["lo"], mid), "mid": mid,
                                             "hi": np.maximum(q["hi"], mid)}, index=pred_df.index)
    return out


def half_width_pct(iv):
    return 100 * (10 ** iv["hi"] - 10 ** iv["lo"]) / (2 * 10 ** iv["mid"])


def score_intervals(ivs, cells):
    """Coverage (with misses below / above), mean half-width, error and width-error correlation."""
    res = []
    for method, iv in ivs.items():
        iv = iv.loc[cells]
        t = y.loc[cells]
        hw, err = half_width_pct(iv), pct_error(t, iv["mid"])
        res.append({"method": method, "n": len(cells),
                    "coverage_pct": 100 * ((t >= iv["lo"]) & (t <= iv["hi"])).mean(),
                    "below_pct (dangerous)": 100 * (t < iv["lo"]).mean(),
                    "above_pct": 100 * (t > iv["hi"]).mean(),
                    "half_width_pct": hw.mean(), "abs_err_pct": err.mean(),
                    "width_vs_error_rho": spearmanr(hw, err)[0]})
    return pd.DataFrame(res).set_index("method")


ivs = intervals(train, data[data["split"] != "train"])
summary = []
for split in ["test", "secondary_test"]:
    s = score_intervals(ivs, data.index[data["split"] == split])
    s.insert(0, "split", split)
    summary.append(s)
pd.concat(summary).round(1)

**Question:** which methods get close to 80% on the primary test, and which on batch 3? For each, are the misses mostly **below** (dangerous) or above? Why is batch conformal wider than policy conformal, and is the extra width worth it on batch 3? With 43 and 40 cells, one cell is ~2.5 points of coverage: which differences are bigger than a few cells?

*Your notes:*

## 4 · Do the intervals flag the hard cells?

From notebook 03: b2c1 (148 cycles, the paper's excluded cell) and the long-lived batch 3 cells the elastic net under-predicted most. A useful interval either **contains** the true life or is **clearly wider than usual** for these cells.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4.4), sharex=True, sharey=True)
lims = [100, 3000]
for ax, (method, iv) in zip(axes, ivs.items()):
    for split, colour in [("test", "C1"), ("secondary_test", "C2")]:
        idx = data.index[data["split"] == split]
        mid, lo, hi = 10 ** iv.loc[idx, "mid"], 10 ** iv.loc[idx, "lo"], 10 ** iv.loc[idx, "hi"]
        ax.errorbar(data.loc[idx, "cycle_life"], mid, yerr=[mid - lo, hi - mid], fmt="o", ms=3,
                    color=colour, elinewidth=0.8, alpha=0.8, label=split)
    ax.plot(lims, lims, "k--", lw=1)
    ax.annotate("b2c1", (data.loc["b2c1", "cycle_life"], 10 ** iv.loc["b2c1", "mid"]),
                xytext=(6, 4), textcoords="offset points", fontsize=8)
    ax.set(xscale="log", yscale="log", xlim=lims, ylim=lims, title=method, xlabel="True cycle life")
axes[0].set_ylabel("Predicted (80% interval)")
axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "04_intervals.png", dpi=200)

hard = [c for c in ["b2c1", "b3c38", "b3c7", "b3c45", "b3c41", "b3c22"] if c in data.index]
rows_hard = []
for method, iv in ivs.items():
    typical = half_width_pct(iv).median()
    for c in hard:
        lo, mid, hi = 10 ** iv.loc[c, ["lo", "mid", "hi"]]
        true = data.loc[c, "cycle_life"]
        rows_hard.append({"method": method, "cell": c, "true": true, "lo": lo, "pred": mid, "hi": hi,
                          "miss": "below (dangerous)" if true < lo else ("above" if true > hi else "inside"),
                          "half_width_pct": 100 * (hi - lo) / (2 * mid), "typical_pct": typical})
pd.DataFrame(rows_hard).set_index(["method", "cell"]).round(0)

**Question:** which methods contain b2c1, and which at least give it a wider-than-typical interval? Why can the tree methods never contain 148 cycles? For the long-lived b3 cells, which direction are the misses, and does that make them dangerous or harmless for a stopping rule?

*Your notes:*

## 5 · Extension: intervals on a new batch

Section 3 used the paper's split, where batch 3 is new but batches 1 and 2 share training. Here each batch is held out in turn and **everything** (models, conformal widths) is built from the other two batches only. For batch conformal, that means the width comes from training on one remaining batch and predicting the other.

If batch-to-batch errors are a fair guide to the next batch, batch conformal should cover close to 80% on every held-out batch.

In [ ]:
cov, lobo_ivs = [], {}
for held_out in BATCHES:
    tr_df, pr_df = data[data["batch"] != held_out], data[data["batch"] == held_out]
    lobo_ivs[held_out] = intervals(tr_df, pr_df)
    s = score_intervals(lobo_ivs[held_out], pr_df.index)
    s.insert(0, "held_out", held_out)
    cov.append(s.reset_index())
cov = pd.concat(cov, ignore_index=True)
cov.pivot(index="held_out", columns="method",
          values=["coverage_pct", "below_pct (dangerous)", "half_width_pct"]).round(1)

**Question:** which method holds coverage best across the three held-out batches, and at what cost in width? Which batch breaks every method, and in which direction (below or above)? Link to the leave-one-batch-out bias in section 1 and notebook 03. Batch conformal learns its width from only **one** batch-to-batch comparison per held-out batch: why does that make it fragile?

*Your notes:*

## 6 · A novelty flag: is this cell unlike the training cells?

Intervals are learned from training cells, so they can't be trusted for a cell far outside them. A simple guard: express each feature in **training standard deviations** (z-scores) and flag a cell if **any** feature is more than 3σ from the training mean. A flagged cell would not be stopped early, whatever its interval says.

(Week 2's outlier review used the same idea: b2c1's fade slope and kurtosis were beyond 4σ.)

In [ ]:
NOVEL_Z = 3.0


def novelty(train_df, pred_df):
    """Largest |z| over the 12 features (training mean/std), and which feature it comes from."""
    mu, sd, med = train_df[F].mean(), train_df[F].std(), train_df[F].median()
    z = ((pred_df[F].fillna(med) - mu) / sd).abs()
    return pd.DataFrame({"max_abs_z": z.max(axis=1), "feature": z.idxmax(axis=1)})


nov = novelty(train, data)
nov["flagged"] = nov["max_abs_z"] > NOVEL_Z
nov["split"], nov["batch"] = data["split"], data["batch"]
nov["EN_err_pct"] = pct_error(y, en_fit.predict(data[F]))

print(nov[nov["split"] != "train"].groupby("split").agg(
    n=("flagged", "size"), flagged=("flagged", "sum"),
    err_flagged=("EN_err_pct", lambda e: e[nov.loc[e.index, "flagged"]].mean()),
    err_not_flagged=("EN_err_pct", lambda e: e[~nov.loc[e.index, "flagged"]].mean())).round(1))
nov[(nov["split"] != "train") & nov["flagged"]].sort_values("max_abs_z", ascending=False).round(1)

**Question:** is b2c1 flagged, and by which feature? Are flagged cells' errors clearly higher than unflagged ones? How many batch 3 cells are flagged, and by which features? Is the flag catching genuinely odd cells or just the batch shift in `ir_min` from week 2? What would a flag that fires on most of a new batch mean for the stopping rule?

*Your notes:*

## 7 · A first stopping rule

At cycle 100, for each cell:

> **Stop the test** if the cell is **not** flagged as novel **and** its 80% interval half-width is at most *W*. Otherwise keep cycling to end of life.

For each method and *W*, count:
- **stopped %**: cells whose test ends at cycle 100;
- **cycles saved %**: cycler time not spent, (true life − 100) summed over stopped cells, as a % of the total for all cells;
- **dangerous**: stopped cells that died **below** the interval's lower bound;
- **harmless misses**: stopped cells that lived beyond the upper bound.

Evaluated two ways: the paper's split (test + secondary) and leave-one-batch-out (every cell predicted by models that never saw its batch). The second is the honest "new cell lot" case.

The two conformal methods give every cell the same width, so for them *W* is all-or-nothing and only the novelty flag selects cells.

In [ ]:
W_OPTIONS = [15, 20, 30, np.inf]


def stop_table(ivs, flags):
    """Apply the rule to every cell in ivs: stop if not flagged and half-width <= W."""
    cells = flags.index
    life, t = data.loc[cells, "cycle_life"], y.loc[cells]
    total = (life - 100).sum()
    res = []
    for method, iv in ivs.items():
        iv = iv.loc[cells]
        hw = half_width_pct(iv)
        for W in W_OPTIONS:
            stop = (~flags) & (hw <= W)
            danger = stop & (t < iv["lo"])
            names = list(danger[danger].index)
            res.append({"method": method, "W_pct": W, "stopped_pct": 100 * stop.mean(),
                        "cycles_saved_pct": 100 * (life[stop] - 100).sum() / total,
                        "dangerous": len(names), "harmless_misses": int((stop & (t > iv["hi"])).sum()),
                        "dangerous_cells": ", ".join(names) if len(names) <= 6 else
                        ", ".join(f"{b}: {n}" for b, n in pd.Series([c[:2] for c in names]).value_counts().items())})
    return pd.DataFrame(res).set_index(["method", "W_pct"])


# (a) paper split: intervals (section 3) and novelty (section 6) from the 41 training cells
cells_eval = data.index[data["split"] != "train"]
paper = stop_table(ivs, nov.loc[cells_eval, "flagged"])

# (b) leave-one-batch-out: join the per-batch intervals from section 5, novelty from the other two batches
ivs_lobo = {m: pd.concat([lobo_ivs[b][m] for b in BATCHES]) for m in ivs}
flags_lobo = pd.concat([novelty(data[data["batch"] != b], data[data["batch"] == b])["max_abs_z"] > NOVEL_Z
                        for b in BATCHES])
lobo_stop = stop_table(ivs_lobo, flags_lobo)

print("(a) Paper split, test + secondary (83 cells):")
print(paper.round(1).to_string())
print("\n(b) Leave-one-batch-out, all 124 cells, each predicted by models that never saw its batch:")
lobo_stop.round(1)

**Question:** for each method, what is the best trade-off you can find between cycles saved and dangerous stops? Which cells are the dangerous ones, and are they the same across methods? How much worse does everything get under leave-one-batch-out than on the paper's split? Write the rule you would actually propose to a test lab, with its expected saving and its expected dangerous-miss rate, and say what you would add before trusting it (link to the k-shot result in notebook 03).

*Your notes:*

---
## Week 4 findings

*Fill in from the outputs above. The drill results (HOML_exercises/ch06) are in the progress doc; note where this notebook agrees or differs.*

- **Tree models (paper split):** tree [ ], forest [ ], boosting [ ] (CV / test / secondary). Best by CV overall: [ ].
- **Leave-one-batch-out:** trees vs elastic net, b1 [ ], b2 [ ], b3 [ ]. Reason: [ ].
- **Permutation importance (held-out, twins together):** above noise for EN: [ ]; for [tree model]: [ ].
- **Intervals, paper split (coverage / dangerous below / ± width):** policy conformal [ ], batch conformal [ ], RF spread [ ], quantile boosting [ ].
- **Hard cells:** b2c1 [ ]; long-lived b3 cells [ ].
- **Intervals on a new batch (leave-one-batch-out):** [ ]. Worst batch [ ], direction [ ].
- **Novelty flag (>3σ):** b2c1 [flagged / not] by [ ]; batch 3 flagged [ ] of 40, mainly by [ ].
- **Stopping rule:** best trade-off [method, W]: [ ]% cycles saved, [ ] dangerous stops (paper split); [ ]% / [ ] under leave-one-batch-out.

**Carry into week 5:** [ ]